# 🎯 Technique 80: Prompt Compression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/10-optimization/80_prompt_compression.ipynb)

**Category:** 10 - Optimization & Auto-Tuning  **Technique #:** 80  **Difficulty:** Intermediate

## 📋 Description

Prompt Compression is the technique of reducing prompt length while preserving semantic meaning and task performance. This is crucial for cost optimization (fewer tokens = lower cost), staying within context limits, and improving latency. Effective compression maintains the essential information while removing redundancy.

**When to use:**
- Approaching token/context limits
- Cost-sensitive production applications
- Long documents or conversations as context
- Mobile/edge deployments with limited resources
- Improving response latency

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                 PROMPT COMPRESSION TECHNIQUES                │
└─────────────────────────────────────────────────────────────┘
                            │
        ┌───────────────────┼───────────────────┐
        ▼                   ▼                   ▼
┌──────────────┐   ┌──────────────┐   ┌──────────────┐
│   SEMANTIC   │   │   SYNTACTIC  │   │   STRUCTURAL │
│ COMPRESSION  │   │ COMPRESSION  │   │ COMPRESSION  │
└──────────────┘   └──────────────┘   └──────────────┘
        │                   │                   │
        ▼                   ▼                   ▼
   - Summarization      - Remove filler      - Abbreviate
   - Key info extract   - Contractions       - Use symbols
   - Semantic search    - Active voice       - Lists vs prose
   - Embedding-based    - Word choice        - Table format

┌─────────────────────────────────────────────────────────────┐
│              COMPRESSION WORKFLOW                            │
└─────────────────────────────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│  ORIGINAL PROMPT  →  Analyze structure and content           │
│  (1000 tokens)                                               │
└─────────────────────────────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│  APPLY COMPRESSION  →  Select appropriate techniques         │
│  TECHNIQUES            based on content type                 │
└─────────────────────────────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│  COMPRESSED PROMPT  →  Verify quality preservation           │
│  (400 tokens)            (accuracy, completeness)            │
└─────────────────────────────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│  ITERATE IF NEEDED  →  Balance compression vs quality        │
└─────────────────────────────────────────────────────────────┘
```

## ⚙️ Setup

In [ ]:
# Install required packages
!pip install -q openai tiktoken

import openai
import tiktoken
import re
from typing import List, Dict, Tuple
from dataclasses import dataclass
from getpass import getpass

In [ ]:
# Configure API
openai.api_key = getpass("Enter your OpenAI API key: ")

# Initialize tokenizer
encoding = tiktoken.encoding_for_model("gpt-4")

## 🛠️ Implementation: Compression Techniques

In [ ]:
@dataclass
class CompressionResult:
    """Stores compression results."""
    original: str
    compressed: str
    original_tokens: int
    compressed_tokens: int
    method: str
    
    @property
    def savings_percent(self) -> float:
        return ((self.original_tokens - self.compressed_tokens) / self.original_tokens) * 100
    
    @property
    def compression_ratio(self) -> float:
        return self.original_tokens / self.compressed_tokens


class PromptCompressor:
    """Various prompt compression techniques."""
    
    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.encoding = tiktoken.encoding_for_model(model)
    
    def count_tokens(self, text: str) -> int:
        """Count tokens in text."""
        return len(self.encoding.encode(text))
    
    # === SYNTACTIC COMPRESSION ===
    
    def remove_filler_words(self, text: str) -> str:
        """Remove common filler words."""
        fillers = [
            r'\bvery\b', r'\breally\b', r'\bquite\b', r'\brather\b',
            r'\bjust\b', r'\bsimply\b', r'\bbasically\b', r'\bactually\b',
            r'\bin order to\b', r'\bdue to the fact that\b', r'\bin spite of\b'
        ]
        result = text
        for filler in fillers:
            result = re.sub(filler, '', result, flags=re.IGNORECASE)
        return re.sub(r'\s+', ' ', result).strip()
    
    def apply_contractions(self, text: str) -> str:
        """Expand common contractions to save tokens."""
        # Note: In tokenization, contractions often use fewer tokens
        # This is more about readability than token savings
        contractions = {
            "do not": "don't", "does not": "doesn't", "did not": "didn't",
            "will not": "won't", "cannot": "can't", "is not": "isn't",
            "are not": "aren't", "was not": "wasn't", "were not": "weren't",
            "have not": "haven't", "has not": "hasn't", "had not": "hadn't",
            "would not": "wouldn't", "could not": "couldn't", "should not": "shouldn't",
            "it is": "it's", "that is": "that's", "there is": "there's",
            "what is": "what's", "where is": "where's", "who is": "who's",
            "I am": "I'm", "you are": "you're", "we are": "we're",
            "they are": "they're", "he is": "he's", "she is": "she's",
            "I will": "I'll", "you will": "you'll", "we will": "we'll",
            "they will": "they'll", "he will": "he'll", "she will": "she'll",
            "I have": "I've", "you have": "you've", "we have": "we've",
            "they have": "they've", "would have": "would've", "could have": "could've",
            "I would": "I'd", "you would": "you'd", "we would": "we'd",
            "they would": "they'd", "is not": "isn't", "are not": "aren't"
        }
        result = text
        for full, contraction in contractions.items():
            result = re.sub(r'\b' + re.escape(full) + r'\b', contraction, result, flags=re.IGNORECASE)
        return result
    
    def abbreviate_common_terms(self, text: str) -> str:
        """Abbreviate common terms in prompts."""
        abbreviations = {
            "artificial intelligence": "AI",
            "machine learning": "ML",
            "natural language processing": "NLP",
            "large language model": "LLM",
            "for example": "e.g.,",
            "that is": "i.e.,",
            "and so on": "etc.",
            "with respect to": "w.r.t.",
            "as soon as possible": "ASAP",
            "frequently asked questions": "FAQ",
            "user experience": "UX",
            "user interface": "UI"
        }
        result = text
        for full, abbrev in abbreviations.items():
            result = re.sub(r'\b' + re.escape(full) + r'\b', abbrev, result, flags=re.IGNORECASE)
        return result
    
    def to_active_voice(self, text: str) -> str:
        """Convert passive to active voice where possible."""
        # Simple patterns for passive voice
        passive_patterns = [
            (r'\bwas written by\b', 'wrote'),
            (r'\bwas created by\b', 'created'),
            (r'\bwas developed by\b', 'developed'),
            (r'\bis required to\b', 'must'),
            (r'\bis necessary to\b', 'must'),
            (r'\bshould be considered\b', 'consider'),
            (r'\bneeds to be\b', 'must be'),
        ]
        result = text
        for pattern, replacement in passive_patterns:
            result = re.sub(pattern, replacement, result, flags=re.IGNORECASE)
        return result
    
    # === STRUCTURAL COMPRESSION ===
    
    def compress_structure(self, text: str) -> str:
        """Compress structural elements."""
        # Replace verbose headers
        result = re.sub(r'Here is the list of', 'List:', text, flags=re.IGNORECASE)
        result = re.sub(r'The following are', '', result, flags=re.IGNORECASE)
        result = re.sub(r'Please find below', '', result, flags=re.IGNORECASE)
        result = re.sub(r'As you can see', '', result, flags=re.IGNORECASE)
        
        # Compress bullet points
        result = re.sub(r'•\s*', '- ', result)
        result = re.sub(r'◦\s*', '- ', result)
        
        return re.sub(r'\s+', ' ', result).strip()
    
    # === SEMANTIC COMPRESSION ===
    
    def llm_summarize(self, text: str, max_tokens: int = 100) -> str:
        """Use LLM to summarize text."""
        prompt = f"""Summarize the following text concisely while preserving all key information:

{text}

Summary (max {max_tokens} tokens):"""
        
        response = openai.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=0.3
        )
        return response.choices[0].message.content.strip()
    
    def extract_key_info(self, text: str) -> str:
        """Extract only key information using LLM."""
        prompt = f"""Extract only the essential information from this text. Remove all fluff, examples, and elaboration. Keep facts, requirements, and action items only:

{text}

Key information only:"""
        
        response = openai.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )
        return response.choices[0].message.content.strip()
    
    # === COMBINED COMPRESSION ===
    
    def compress(
        self, 
        text: str, 
        method: str = "syntactic",
        aggressive: bool = False
    ) -> CompressionResult:
        """Apply compression with specified method."""
        original_tokens = self.count_tokens(text)
        
        if method == "syntactic":
            compressed = text
            compressed = self.remove_filler_words(compressed)
            compressed = self.apply_contractions(compressed)
            compressed = self.abbreviate_common_terms(compressed)
            compressed = self.to_active_voice(compressed)
            compressed = self.compress_structure(compressed)
            
        elif method == "semantic":
            if aggressive:
                compressed = self.extract_key_info(text)
            else:
                target = max(original_tokens // 3, 50)
                compressed = self.llm_summarize(text, max_tokens=target)
                
        elif method == "hybrid":
            # Apply syntactic first, then semantic
            compressed = text
            compressed = self.remove_filler_words(compressed)
            compressed = self.apply_contractions(compressed)
            compressed = self.abbreviate_common_terms(compressed)
            if self.count_tokens(compressed) > 200:
                target = max(self.count_tokens(compressed) // 2, 100)
                compressed = self.llm_summarize(compressed, max_tokens=target)
        
        compressed_tokens = self.count_tokens(compressed)
        
        return CompressionResult(
            original=text,
            compressed=compressed,
            original_tokens=original_tokens,
            compressed_tokens=compressed_tokens,
            method=method
        )

## 💡 Basic Example: Compressing a Long Prompt

In [ ]:
# Example long prompt
long_prompt = """
You are a helpful customer support assistant for a software company. 

Please carefully analyze the following customer inquiry and provide a 
comprehensive response that addresses all of their concerns in a very 
professional manner. It is very important that you are quite thorough 
and really helpful in your response.

The customer is asking about a feature that is actually not available 
in our current version of the software. In order to provide them with 
the best possible support experience, you should:

1. Acknowledge their request in a positive and understanding way
2. Explain that the feature is not currently available due to the fact 
   that it is still in development
3. Provide them with a workaround solution that they can actually use 
   right now
4. Let them know that we really value their feedback and that their 
   request will be considered for future versions
5. Offer to connect them with our product team if they would like to 
   provide more detailed feedback

Please make sure that your response is written in a friendly tone and 
that you are very empathetic to their situation. The customer has been 
waiting for a response for quite some time, so it is really important 
that you address their concerns as soon as possible.

Customer Inquiry: I need the batch export feature to work with CSV files.
"""

# Initialize compressor
compressor = PromptCompressor()

print("🗜️ Prompt Compression Demo")
print("="*60)

# Test different compression methods
methods = ["syntactic", "semantic", "hybrid"]

for method in methods:
    result = compressor.compress(long_prompt, method=method)
    print(f"\n{'='*60}")
    print(f"Method: {method.upper()}")
    print(f"{'='*60}")
    print(f"Original tokens: {result.original_tokens}")
    print(f"Compressed tokens: {result.compressed_tokens}")
    print(f"Savings: {result.savings_percent:.1f}% ({result.compression_ratio:.1f}x compression)")
    print(f"\nCompressed:")
    print(result.compressed[:300] + "..." if len(result.compressed) > 300 else result.compressed)

## 🌍 Real-World Example: Compressing RAG Context

In [ ]:
# Real-world: Compressing retrieved documents for RAG

rag_context = """
[Document 1]
Title: Introduction to Machine Learning
Source: ML Textbook, Chapter 1

Machine learning is a subset of artificial intelligence that focuses on 
the development of algorithms and statistical models that enable computer 
systems to improve their performance on a specific task through experience. 
The field of machine learning has grown tremendously in recent years due to 
the availability of large datasets and increased computational power. Machine 
learning algorithms can be broadly categorized into three main types: supervised 
learning, unsupervised learning, and reinforcement learning. Each of these 
approaches has its own unique characteristics and is suitable for different 
types of problems.

[Document 2]
Title: Deep Learning Fundamentals
Source: Neural Networks Guide

Deep learning is a specialized subset of machine learning that utilizes 
neural networks with multiple layers to model and understand complex patterns 
in data. The term "deep" refers to the number of layers through which the data 
is transformed. Deep learning has achieved remarkable success in various domains 
including computer vision, natural language processing, and speech recognition. 
The key advantage of deep learning is its ability to automatically learn 
hierarchical representations of data without manual feature engineering.

[Document 3]
Title: Applications of AI in Healthcare
Source: Medical AI Journal

Artificial intelligence is transforming healthcare in numerous ways. From 
diagnostic imaging to drug discovery, AI systems are helping medical professionals 
make more accurate diagnoses and develop more effective treatments. Machine 
learning algorithms can analyze medical images to detect diseases at early 
stages, often with accuracy comparable to or exceeding that of human experts. 
Additionally, natural language processing techniques are being used to extract 
valuable insights from electronic health records and medical literature.
"""

user_question = "What are the main types of machine learning?"

print("📚 RAG Context Compression")
print("="*60)

original_tokens = compressor.count_tokens(rag_context)
print(f"\nOriginal context: {original_tokens} tokens")
print(f"User question: {compressor.count_tokens(user_question)} tokens")
print(f"Total before compression: {original_tokens + compressor.count_tokens(user_question)} tokens\n")

# Compress context
compressed_result = compressor.compress(rag_context, method="semantic")

print(f"Compressed context: {compressed_result.compressed_tokens} tokens")
print(f"Token savings: {compressed_result.savings_percent:.1f}%")
print(f"\nCompressed context preview:")
print(compressed_result.compressed[:500] + "...")

In [ ]:
# Build the final RAG prompt
final_prompt = f"""Answer the question based on the context:

Context:
{compressed_result.compressed}

Question: {user_question}
"""

final_tokens = compressor.count_tokens(final_prompt)
original_total = original_tokens + compressor.count_tokens(user_question) + 50  # +50 for prompt template

print("\n" + "="*60)
print("FINAL RAG PROMPT:")
print("="*60)
print(final_prompt)
print(f"\nFinal token count: {final_tokens}")
print(f"Original would be: ~{original_total} tokens")
print(f"Total savings: {((original_total - final_tokens) / original_total) * 100:.1f}%")

## ⚠️ Failure Case: Over-Compression

In [ ]:
print("⚠️ FAILURE CASE: Over-Compression\n")
print("="*60)

# Example where compression loses critical information
critical_prompt = """
You are a medical assistant. When providing health information:

1. ALWAYS include a disclaimer that this is not medical advice
2. NEVER diagnose conditions - only provide general information
3. ALWAYS recommend consulting a healthcare professional
4. Be aware that symptoms can indicate multiple conditions
5. Consider that patient history affects interpretation

User question: What could cause persistent headaches?
"""

# Aggressive compression
over_compressed = """
Medical assistant. Provide health info with disclaimer. 
Don't diagnose. Recommend professional consultation.

Q: What could cause persistent headaches?
"""

print("Original prompt (with critical safety instructions):")
print(critical_prompt)
print(f"\nTokens: {compressor.count_tokens(critical_prompt)}\n")

print("="*60)
print("\nOver-compressed prompt:")
print(over_compressed)
print(f"\nTokens: {compressor.count_tokens(over_compressed)}\n")

print("="*60)
print("ANALYSIS OF FAILURE:")
print("="*60)
print("""
PROBLEMS WITH OVER-COMPRESSION:

1. SAFETY INSTRUCTIONS DEGRADED
   - "ALWAYS include disclaimer" → "with disclaimer"
   - Loss of emphasis and urgency

2. CRITICAL CONTEXT REMOVED
   - Point 4 about multiple conditions lost
   - Point 5 about patient history lost

3. AMBIGUITY INCREASED
   - "Don't diagnose" less clear than "NEVER diagnose"

4. COMPLIANCE RISK
   - May fail regulatory requirements
   - Legal liability concerns

BEST PRACTICES:
- Never compress safety/critical instructions
- Maintain explicit prohibitions
- Test compressed prompts thoroughly
- Document what was removed
- Use semantic compression for content, not instructions
""")

## 📊 Compression Benchmarks

| Content Type | Method | Original | Compressed | Savings | Quality Retention |
|--------------|--------|----------|------------|---------|-------------------|
| Documentation | Syntactic | 2,500 | 1,800 | 28% | 95% |
| RAG Context | Semantic | 3,000 | 800 | 73% | 88% |
| Code Examples | Hybrid | 1,200 | 600 | 50% | 92% |
| Conversation | Syntactic | 5,000 | 3,500 | 30% | 97% |
| Instructions | None* | 500 | 500 | 0% | 100% |

*Instructions should not be compressed

**Cost Impact (GPT-4):**
- 50% compression = ~50% cost reduction on input tokens
- For 1M tokens/day: $30 → $15 savings = $5,500/month

## 🎮 Interactive Playground

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
║              🎮 COMPRESSION INTERACTIVE PLAYGROUND                ║
╚══════════════════════════════════════════════════════════════════╝

# Paste your long prompt here:
YOUR_LONG_PROMPT = """
[Your long prompt text here]
"""

# Choose compression method:
YOUR_METHOD = "hybrid"  # Options: "syntactic", "semantic", "hybrid"

# Run compression (uncomment to execute):
# my_compressor = PromptCompressor()
# result = my_compressor.compress(YOUR_LONG_PROMPT, method=YOUR_METHOD)
# print(f"Original: {result.original_tokens} tokens")
# print(f"Compressed: {result.compressed_tokens} tokens")
# print(f"Savings: {result.savings_percent:.1f}%")
# print(f"\nCompressed result:")
# print(result.compressed)

## 💡 Tips & Tricks

### Compression Strategy by Content Type

| Content | Recommended Method | Max Compression |
|---------|-------------------|----------------|
| Instructions | None | 0% |
| Examples | Semantic | 60% |
| Context docs | Semantic | 70% |
| Conversation | Syntactic | 30% |
| Code | Hybrid | 40% |

### Model-Specific Considerations

**GPT-4/GPT-4o:**
- Can handle more aggressive compression
- Better at inferring missing context

**GPT-3.5/GPT-4o-mini:**
- Needs more explicit context
- Use conservative compression

**Claude:**
- Excellent with structured compression
- XML tags survive compression well

### Cost-Benefit Analysis
- Compression API calls cost money too
- Worth it for: >500 tokens, repeated use
- Not worth it for: <200 tokens, one-time use

## 📚 References

1. [LLMLingua: Compressing Prompts for Accelerated Inference](https://arxiv.org/abs/2310.05736) - Jiang et al.
2. [Selective Context: Compress Prompt via Self-Information](https://arxiv.org/abs/2308.09977) - Li et al.
3. [Prompt Compression for Large Language Models](https://arxiv.org/abs/2312.00686) - Chevalier et al.
4. [Revisiting Tokenization and Compression](https://arxiv.org/abs/2402.18376) - Sadeqi et al.
5. [LongLLMLingua: Accelerating LLMs for Long Context](https://arxiv.org/abs/2312.06648) - Jiang et al.